# Data Integration Notebook

## Objective
This notebook manages the integration of all datasets (internal and external) for the **Triagegeist** competition.

### What this notebook does:
1. **Reorganizes the folder structure** - moves existing datasets into the subfolder `dataset/raw/triagegeist/`
2. **Reproduces the merge** of the 4 original CSVs (as done in `notebook.ipynb`)
3. **Defines the target schema** - a JSON file describing the format that every dataset (internal and external) must conform to
4. **Saves the cleaned Triagegeist dataset** in `dataset/processed/`
5. *(Future sections)* Cleaning and harmonization of MIMIC-IV-ED, NHAMCS, UCI

### Final folder structure:
```
dataset/
├── raw/                        # original data, NEVER modified
│   ├── triagegeist/            # Kaggle competition CSVs
│   ├── mimic_iv_ed/            # 
│   ├── nhamcs/                 # 
│   └── uci/                    # 
├── processed/                  # cleaned datasets, one per source
│   ├── triagegeist_clean.csv
│   ├── mimic_clean.csv         # (future)
│   ├── nhamcs_clean.csv        # (future)
│   └── uci_clean.csv           # (future)
├── merged/                     # final unified dataset
│   └── train_merged.csv
└── schema_target.json          # reference schema
```

In [1]:
import pandas as pd
import numpy as np
import json
import os
import shutil

---
# Section 1: Folder Structure Reorganization

We move the original competition CSVs into `dataset/raw/triagegeist/` and create all necessary subfolders.

> **Note**: this cell should be run **only once**. After the first execution, files will already be in place and the cell will report accordingly.

In [2]:
# === Folder definitions ===
BASE_DIR = "dataset"
RAW_DIR = os.path.join(BASE_DIR, "raw")
TRIAGEGEIST_RAW = os.path.join(RAW_DIR, "triagegeist")
MIMIC_RAW = os.path.join(RAW_DIR, "mimic_iv_ed")
NHAMCS_RAW = os.path.join(RAW_DIR, "nhamcs")
UCI_RAW = os.path.join(RAW_DIR, "uci")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
MERGED_DIR = os.path.join(BASE_DIR, "merged")

# Create all subfolders if they don't exist
for folder in [TRIAGEGEIST_RAW, MIMIC_RAW, NHAMCS_RAW, UCI_RAW, PROCESSED_DIR, MERGED_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"Folder ready: {folder}")

# === Move original CSVs to dataset/raw/triagegeist/ ===
# These are the original Kaggle competition files.
# We move them ONLY if they are still in the dataset/ root folder.

ORIGINAL_FILES = [
    "train.csv",
    "test.csv",
    "chief_complaints.csv",
    "patient_history.csv",
    "sample_submission.csv",
]

# Also move files generated by the merge in notebook.ipynb, for cleanup
GENERATED_FILES = [
    "train_dataset.csv",
    "test_dataset.csv",
]

moved_count = 0
for filename in ORIGINAL_FILES + GENERATED_FILES:
    src = os.path.join(BASE_DIR, filename)
    dst = os.path.join(TRIAGEGEIST_RAW, filename)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.move(src, dst)
        print(f"Moved: {src} -> {dst}")
        moved_count += 1
    elif os.path.exists(dst):
        print(f"Already present: {dst}")
    else:
        print(f"Not found: {src} (may have already been moved)")

print(f"\nMove completed: {moved_count} files moved.")

Folder ready: dataset\raw\triagegeist
Folder ready: dataset\raw\mimic_iv_ed
Folder ready: dataset\raw\nhamcs
Folder ready: dataset\raw\uci
Folder ready: dataset\processed
Folder ready: dataset\merged
Already present: dataset\raw\triagegeist\train.csv
Already present: dataset\raw\triagegeist\test.csv
Already present: dataset\raw\triagegeist\chief_complaints.csv
Already present: dataset\raw\triagegeist\patient_history.csv
Already present: dataset\raw\triagegeist\sample_submission.csv
Already present: dataset\raw\triagegeist\train_dataset.csv
Already present: dataset\raw\triagegeist\test_dataset.csv

Move completed: 0 files moved.


---
# Section 2: Loading and Merging Triagegeist Data

We reproduce exactly the merge done in `notebook.ipynb`:
1. Load the 4 original CSVs from `dataset/raw/triagegeist/`
2. Join `chief_complaints` + `patient_history` with an **outer join** on `patient_id`
3. Join the result with `train.csv` and `test.csv` with a **left join** on `patient_id`
4. Remove `chief_complaint_system` 

In [3]:
# === Load the 4 original CSVs ===
train_df = pd.read_csv(os.path.join(TRIAGEGEIST_RAW, "train.csv"))
test_df = pd.read_csv(os.path.join(TRIAGEGEIST_RAW, "test.csv"))
complaints_df = pd.read_csv(os.path.join(TRIAGEGEIST_RAW, "chief_complaints.csv"))
history_df = pd.read_csv(os.path.join(TRIAGEGEIST_RAW, "patient_history.csv"))

print(f"train.csv:             {train_df.shape}")
print(f"test.csv:              {test_df.shape}")
print(f"chief_complaints.csv:  {complaints_df.shape}")
print(f"patient_history.csv:   {history_df.shape}")

train.csv:             (80000, 40)
test.csv:              (20000, 37)
chief_complaints.csv:  (100000, 3)
patient_history.csv:   (100000, 26)


In [4]:
# === Merge patient information ===
# outer join: keep all patients from both tables
patient_info_df = complaints_df.merge(
    history_df,
    on="patient_id",
    how="outer"
)
print(f"patient_info after outer join: {patient_info_df.shape}")

# === Remove redundant column ===
# chief_complaint_system is partially redundant with the raw text chief_complaint_raw
for df in [train_df, test_df]:
    if "chief_complaint_system" in df.columns:
        df.drop(columns=["chief_complaint_system"], inplace=True)

# === Left join: enrich train and test with patient info ===
# left join: keep all rows from train/test, add info if available
train_full_df = train_df.merge(patient_info_df, on="patient_id", how="left")
test_full_df = test_df.merge(patient_info_df, on="patient_id", how="left")

print(f"\ntrain after merge: {train_full_df.shape}")
print(f"test after merge:  {test_full_df.shape}")

# === Verify: columns present only in train ===
train_only_cols = [c for c in train_full_df.columns if c not in test_full_df.columns]
print(f"\nColumns present ONLY in train (target + outcome): {train_only_cols}")

patient_info after outer join: (100000, 28)

train after merge: (80000, 66)
test after merge:  (20000, 63)

Columns present ONLY in train (target + outcome): ['disposition', 'ed_los_hours', 'triage_acuity']


---
# Section 3: Target Schema Definition

The target schema defines the "contract" that every dataset (internal and external) must conform to.

For each column we specify:
- **name**: column name in the final format
- **dtype**: data type (`categorical`, `numerical`, `binary`, `text`)
- **role**: role in the model (`feature`, `target`, `outcome_only`, `id`, `text_raw`)
- **range**: allowed values or valid interval
- **required**: whether the column is mandatory or can be all NaN (for external datasets missing that feature)
- **notes**: notes about the column

> External datasets that lack a given feature will have that column filled with NaN. The model will handle imputation during preprocessing.

In [5]:
schema_target = {
    "description": "Target schema for the Triagegeist project. Every dataset (internal and external) must be transformed to conform to this format.",
    "version": "1.0",
    "target_column": "triage_acuity",
    "id_column": "patient_id",
    
    "columns": {
        # === IDENTIFIER ===
        "patient_id": {
            "dtype": "string",
            "role": "id",
            "required": True,
            "notes": "Unique patient identifier. For external datasets, generate an ID with a source prefix (e.g. MIMIC_12345)"
        },
        
        # === TARGET ===
        "triage_acuity": {
            "dtype": "integer",
            "role": "target",
            "range": [1, 5],
            "required": True,
            "notes": "ESI scale: 1=immediate, 2=emergent, 3=urgent/multiple resources, 4=less urgent/one resource, 5=nonurgent/no resources"
        },
        
        # === OUTCOME (validation only, do NOT use as feature) ===
        "disposition": {
            "dtype": "categorical",
            "role": "outcome_only",
            "values": ["discharged", "admitted", "transferred", "observation", "deceased", "lwbs", "lama"],
            "required": False,
            "notes": "Visit outcome. Do NOT use in training - causes data leakage."
        },
        "ed_los_hours": {
            "dtype": "float",
            "role": "outcome_only",
            "range": [0, 48],
            "required": False,
            "notes": "ED length of stay (hours). Do NOT use in training."
        },

        # === CATEGORICAL FEATURES ===
        "arrival_mode": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["walk-in", "ambulance", "transfer", "police", "helicopter", "other"],
            "required": False,
            "notes": "Mode of arrival to ED"
        },
        "arrival_day": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"],
            "required": False,
            "notes": "Day of the week of arrival"
        },
        "arrival_season": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["spring", "summer", "autumn", "winter"],
            "required": False,
            "notes": "Season of arrival"
        },
        "shift": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["morning", "afternoon", "evening", "night"],
            "required": False,
            "notes": "Shift during which the patient arrived"
        },
        "age_group": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["pediatric", "young_adult", "middle_aged", "senior"],
            "required": False,
            "notes": "Age bracket. Can be derived from age if not present."
        },
        "sex": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["M", "F", "Other"],
            "required": True,
            "notes": "Patient sex"
        },
        "language": {
            "dtype": "categorical",
            "role": "feature",
            "required": False,
            "notes": "Patient primary language"
        },
        "insurance_type": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["public", "private", "self-pay", "military", "other"],
            "required": False,
            "notes": "Insurance type. ~60% public in the Triagegeist dataset."
        },
        "transport_origin": {
            "dtype": "categorical",
            "role": "feature",
            "required": False,
            "notes": "Transport origin (home, workplace, street, etc.)"
        },
        "pain_location": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["chest", "abdomen", "head", "limb", "pelvis", "back", "neck", "diffuse", "none"],
            "required": False,
            "notes": "Reported pain location"
        },
        "mental_status_triage": {
            "dtype": "categorical",
            "role": "feature",
            "values": ["alert", "confused", "lethargic", "obtunded", "unresponsive"],
            "required": False,
            "notes": "Mental status at triage"
        },
        "site_id": {
            "dtype": "categorical",
            "role": "feature",
            "required": False,
            "notes": "Hospital identifier. For external datasets, use a source-specific code."
        },
        "triage_nurse_id": {
            "dtype": "categorical",
            "role": "feature",
            "required": False,
            "notes": "Triage nurse ID. Useful for inter-rater bias analysis."
        },

        # === NUMERICAL FEATURES - VITAL SIGNS ===
        "systolic_bp": {
            "dtype": "float",
            "role": "feature",
            "range": [30, 260],
            "unit": "mmHg",
            "required": False,
            "notes": "Systolic blood pressure. ~5% missing in Triagegeist."
        },
        "diastolic_bp": {
            "dtype": "float",
            "role": "feature",
            "range": [10, 160],
            "unit": "mmHg",
            "required": False,
            "notes": "Diastolic blood pressure"
        },
        "mean_arterial_pressure": {
            "dtype": "float",
            "role": "feature",
            "range": [20, 180],
            "unit": "mmHg",
            "required": False,
            "notes": "MAP = (SBP + 2*DBP) / 3. Can be derived if not present."
        },
        "pulse_pressure": {
            "dtype": "float",
            "role": "feature",
            "range": [-60, 180],
            "unit": "mmHg",
            "required": False,
            "notes": "SBP - DBP. Can be derived if not present."
        },
        "heart_rate": {
            "dtype": "float",
            "role": "feature",
            "range": [20, 250],
            "unit": "bpm",
            "required": False,
            "notes": "Heart rate"
        },
        "respiratory_rate": {
            "dtype": "float",
            "role": "feature",
            "range": [4, 60],
            "unit": "breaths/min",
            "required": False,
            "notes": "Respiratory rate. ~3.8% missing."
        },
        "temperature_c": {
            "dtype": "float",
            "role": "feature",
            "range": [32, 44],
            "unit": "Celsius",
            "required": False,
            "notes": "Core body temperature in Celsius. WARNING: NHAMCS uses Fahrenheit - must convert!"
        },
        "spo2": {
            "dtype": "float",
            "role": "feature",
            "range": [50, 100],
            "unit": "%",
            "required": False,
            "notes": "Peripheral oxygen saturation"
        },
        "gcs_total": {
            "dtype": "integer",
            "role": "feature",
            "range": [3, 15],
            "required": False,
            "notes": "Glasgow Coma Scale total. 15=alert, 3=deep coma."
        },
        "pain_score": {
            "dtype": "integer",
            "role": "feature",
            "range": [-1, 10],
            "required": False,
            "notes": "Self-reported pain. -1 = not available, 0 = no pain, 10 = worst."
        },

        # === NUMERICAL FEATURES - PHYSICAL MEASUREMENTS ===
        "weight_kg": {
            "dtype": "float",
            "role": "feature",
            "range": [1, 300],
            "unit": "kg",
            "required": False,
            "notes": "Weight in kg"
        },
        "height_cm": {
            "dtype": "float",
            "role": "feature",
            "range": [30, 230],
            "unit": "cm",
            "required": False,
            "notes": "Height in cm"
        },
        "bmi": {
            "dtype": "float",
            "role": "feature",
            "range": [5, 80],
            "required": False,
            "notes": "Body Mass Index. Can be derived from weight and height."
        },
        "shock_index": {
            "dtype": "float",
            "role": "feature",
            "range": [0.1, 5.0],
            "required": False,
            "notes": "HR / SBP. >0.9 suggests shock state. Can be derived."
        },
        "news2_score": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 20],
            "required": False,
            "notes": "National Early Warning Score 2. Composite severity score."
        },

        # === NUMERICAL FEATURES - COUNTS AND TEMPORAL ===
        "age": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 120],
            "required": True,
            "notes": "Patient age in years"
        },
        "arrival_hour": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 23],
            "required": False,
            "notes": "Arrival hour (0-23)"
        },
        "arrival_month": {
            "dtype": "integer",
            "role": "feature",
            "range": [1, 12],
            "required": False,
            "notes": "Arrival month (1-12)"
        },
        "num_prior_ed_visits_12m": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 50],
            "required": False,
            "notes": "ED visits in the past 12 months"
        },
        "num_prior_admissions_12m": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 50],
            "required": False,
            "notes": "Hospital admissions in the past 12 months"
        },
        "num_active_medications": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 50],
            "required": False,
            "notes": "Number of active medications"
        },
        "num_comorbidities": {
            "dtype": "integer",
            "role": "feature",
            "range": [0, 30],
            "required": False,
            "notes": "Number of comorbidities"
        },

        # === BINARY FEATURES - MEDICAL HISTORY ===
        "hx_hypertension":              {"dtype": "binary", "role": "feature", "required": False, "notes": "Hypertension"},
        "hx_diabetes_type2":            {"dtype": "binary", "role": "feature", "required": False, "notes": "Type 2 diabetes"},
        "hx_diabetes_type1":            {"dtype": "binary", "role": "feature", "required": False, "notes": "Type 1 diabetes"},
        "hx_asthma":                    {"dtype": "binary", "role": "feature", "required": False, "notes": "Asthma"},
        "hx_copd":                      {"dtype": "binary", "role": "feature", "required": False, "notes": "COPD"},
        "hx_heart_failure":             {"dtype": "binary", "role": "feature", "required": False, "notes": "Heart failure"},
        "hx_atrial_fibrillation":       {"dtype": "binary", "role": "feature", "required": False, "notes": "Atrial fibrillation"},
        "hx_ckd":                       {"dtype": "binary", "role": "feature", "required": False, "notes": "Chronic kidney disease"},
        "hx_liver_disease":             {"dtype": "binary", "role": "feature", "required": False, "notes": "Liver disease"},
        "hx_malignancy":                {"dtype": "binary", "role": "feature", "required": False, "notes": "Active/prior malignancy"},
        "hx_obesity":                   {"dtype": "binary", "role": "feature", "required": False, "notes": "Obesity (BMI>30)"},
        "hx_depression":                {"dtype": "binary", "role": "feature", "required": False, "notes": "Depression"},
        "hx_anxiety":                   {"dtype": "binary", "role": "feature", "required": False, "notes": "Anxiety"},
        "hx_dementia":                  {"dtype": "binary", "role": "feature", "required": False, "notes": "Dementia"},
        "hx_epilepsy":                  {"dtype": "binary", "role": "feature", "required": False, "notes": "Epilepsy"},
        "hx_hypothyroidism":            {"dtype": "binary", "role": "feature", "required": False, "notes": "Hypothyroidism"},
        "hx_hyperthyroidism":           {"dtype": "binary", "role": "feature", "required": False, "notes": "Hyperthyroidism"},
        "hx_hiv":                       {"dtype": "binary", "role": "feature", "required": False, "notes": "HIV"},
        "hx_coagulopathy":              {"dtype": "binary", "role": "feature", "required": False, "notes": "Coagulopathy"},
        "hx_immunosuppressed":          {"dtype": "binary", "role": "feature", "required": False, "notes": "Immunosuppressed"},
        "hx_pregnant":                  {"dtype": "binary", "role": "feature", "required": False, "notes": "Pregnancy"},
        "hx_substance_use_disorder":    {"dtype": "binary", "role": "feature", "required": False, "notes": "Substance use disorder"},
        "hx_coronary_artery_disease":   {"dtype": "binary", "role": "feature", "required": False, "notes": "Coronary artery disease"},
        "hx_stroke_prior":              {"dtype": "binary", "role": "feature", "required": False, "notes": "Prior stroke"},
        "hx_peripheral_vascular_disease":{"dtype": "binary", "role": "feature", "required": False, "notes": "Peripheral vascular disease"},

        # === RAW TEXT ===
        "chief_complaint_raw": {
            "dtype": "text",
            "role": "text_raw",
            "required": False,
            "notes": "Free-text chief complaint. To be processed with NLP."
        },

        # === SOURCE METADATA ===
        "data_source": {
            "dtype": "categorical",
            "role": "metadata",
            "values": ["triagegeist", "mimic_iv_ed", "nhamcs", "uci"],
            "required": True,
            "notes": "Indicates which dataset the record comes from. Useful for analysis and debugging."
        }
    }
}

# === Save schema to file ===
schema_path = os.path.join(BASE_DIR, "schema_target.json")
with open(schema_path, "w") as f:
    json.dump(schema_target, f, indent=2)

print(f"Target schema saved to: {schema_path}")
print(f"Total number of columns in schema: {len(schema_target['columns'])}")
print(f"\nRequired columns (required=True):")
for name, info in schema_target['columns'].items():
    if info.get('required'):
        print(f"  - {name} ({info['dtype']}, {info['role']})")

Target schema saved to: dataset\schema_target.json
Total number of columns in schema: 66

Required columns (required=True):
  - patient_id (string, id)
  - triage_acuity (integer, target)
  - sex (categorical, feature)
  - age (integer, feature)
  - data_source (categorical, metadata)


---
# Section 4: Triagegeist Dataset Cleaning and Saving to processed/

We take the already-merged dataset and make it conform to the target schema:
- Add the `data_source` column
- Keep only columns present in the schema
- Validate that ranges are respected
- Save to `dataset/processed/triagegeist_clean.csv`

In [6]:
def validate_and_clean(df, schema, source_name):
    """
    Takes a DataFrame and makes it conform to the target schema.
    - Adds data_source
    - Selects only schema columns (adds missing ones with NaN)
    - Validates numerical ranges
    
    Returns:
        df_clean: DataFrame conforming to the schema
        report: dict with validation statistics
    """
    schema_cols = list(schema['columns'].keys())
    report = {"source": source_name, "issues": []}
    
    # 1. Add data_source
    df = df.copy()
    df['data_source'] = source_name
    
    # 2. Columns present in df but NOT in schema (will be dropped)
    extra_cols = [c for c in df.columns if c not in schema_cols]
    if extra_cols:
        report['issues'].append(f"Dropped columns (not in schema): {extra_cols}")
    
    # 3. Columns in schema but NOT in df (will be created with NaN)
    missing_cols = [c for c in schema_cols if c not in df.columns]
    if missing_cols:
        report['issues'].append(f"Missing columns (created with NaN): {missing_cols}")
        for col in missing_cols:
            df[col] = np.nan
    
    # 4. Select only schema columns, in schema order
    df_clean = df[schema_cols].copy()
    
    # 5. Range validation for numerical columns
    range_violations = {}
    for col_name, col_info in schema['columns'].items():
        if 'range' in col_info and col_name in df_clean.columns:
            col_data = df_clean[col_name].dropna()
            if len(col_data) == 0:
                continue
            lo, hi = col_info['range']
            out_of_range = ((col_data < lo) | (col_data > hi)).sum()
            if out_of_range > 0:
                range_violations[col_name] = {
                    "n_violations": int(out_of_range),
                    "expected_range": [lo, hi],
                    "actual_min": float(col_data.min()),
                    "actual_max": float(col_data.max())
                }
    if range_violations:
        report['issues'].append(f"Range violations: {range_violations}")
    
    report['final_shape'] = df_clean.shape
    report['nan_percentage'] = round(df_clean.isna().mean().mean() * 100, 2)
    
    return df_clean, report

print("Function validate_and_clean() defined.")

Function validate_and_clean() defined.


In [7]:
# === Clean Triagegeist train set ===
triagegeist_clean, triagegeist_report = validate_and_clean(
    train_full_df, schema_target, "triagegeist"
)

print("=== Triagegeist Validation Report (train) ===")
print(f"Final shape: {triagegeist_report['final_shape']}")
print(f"Average NaN per column: {triagegeist_report['nan_percentage']}%")
for issue in triagegeist_report['issues']:
    print(f"  - {issue}")

# === Save ===
output_path = os.path.join(PROCESSED_DIR, "triagegeist_clean.csv")
triagegeist_clean.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
print(f"Shape: {triagegeist_clean.shape}")
triagegeist_clean.head(3)

=== Triagegeist Validation Report (train) ===
Final shape: (80000, 66)
Average NaN per column: 0.46%
  - Dropped columns (not in schema): ['chief_complaint_system']

Saved to: dataset\processed\triagegeist_clean.csv
Shape: (80000, 66)


,patient_id,triage_acuity,disposition,ed_los_hours,arrival_mode,arrival_day,arrival_season,shift,age_group,sex,...,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease,chief_complaint_raw,data_source
0,TG-UXRGA9UCO,2,discharged,7.35,walk-in,Monday,spring,morning,middle_aged,M,...,0,0,1,0,0,0,0,0,"thunderclap headache, worsening with movement",triagegeist
1,TG-B19DBBS2G,5,discharged,0.70,walk-in,Thursday,spring,morning,elderly,F,...,0,0,1,0,1,1,0,0,"contraception advice, intermittent",triagegeist
2,TG-GZ97W7M6V,5,discharged,0.63,walk-in,Saturday,spring,morning,elderly,M,...,1,0,0,0,0,1,1,1,"general health question, intermittent",triagegeist


---
# Section 5: Triagegeist Test Set Cleaning

> **IMPORTANT**: the Kaggle test set should never be modified in substance and should never be enriched with external data. External data is used **only for training**. The test set is simply made to conform to the schema for pipeline consistency.

In [8]:
# === Clean Triagegeist test set ===
triagegeist_test_clean, test_report = validate_and_clean(
    test_full_df, schema_target, "triagegeist"
)

print("=== Triagegeist Validation Report (test) ===")
print(f"Final shape: {test_report['final_shape']}")
print(f"Average NaN per column: {test_report['nan_percentage']}%")
for issue in test_report['issues']:
    print(f"  - {issue}")

# === Save ===
test_output_path = os.path.join(PROCESSED_DIR, "triagegeist_test_clean.csv")
triagegeist_test_clean.to_csv(test_output_path, index=False)
print(f"\nSaved to: {test_output_path}")

=== Triagegeist Validation Report (test) ===
Final shape: (20000, 66)
Average NaN per column: 4.97%
  - Dropped columns (not in schema): ['chief_complaint_system']
  - Missing columns (created with NaN): ['triage_acuity', 'disposition', 'ed_los_hours']

Saved to: dataset\processed\triagegeist_test_clean.csv


---
# Section 6: Summary and Next Steps

## What we have done
- Reorganized files into `dataset/raw/triagegeist/`
- Reproduced the merge of the 4 original CSVs
- Defined and saved the target schema (`schema_target.json`)
- Cleaned and validated Triagegeist train and test sets, saved in `dataset/processed/`

## Next steps
For each external dataset, create a dedicated section that:
1. Loads raw data from `dataset/raw/<source>/`
2. Renames columns to match the target schema
3. Converts units of measurement (e.g. Fahrenheit -> Celsius)
4. Maps categories to schema values
5. Calls `validate_and_clean()` to conform to the schema
6. Saves to `dataset/processed/<source>_clean.csv`

At the end, a **final merge** section will concatenate all processed files into a single `dataset/merged/train_merged.csv`.

### Template for adding a new dataset:
```python
# === Load raw data ===
raw_df = pd.read_csv(os.path.join(NEW_SOURCE_RAW, "file.csv"))

# === Rename columns ===
rename_map = {
    "old_name": "schema_target_name",
    # ...
}
raw_df.rename(columns=rename_map, inplace=True)

# === Conversions ===
raw_df['temperature_c'] = (raw_df['temperature_f'] - 32) * 5/9  # example F->C

# === Map categories ===
raw_df['sex'] = raw_df['sex'].map({'MALE': 'M', 'FEMALE': 'F'})

# === Validate and save ===
clean_df, report = validate_and_clean(raw_df, schema_target, "source_name")
clean_df.to_csv(os.path.join(PROCESSED_DIR, "source_name_clean.csv"), index=False)
```

------

# NHAMCS Section - Loading and Cleaning Data 2017-2022

## What is NHAMCS
The **National Hospital Ambulatory Medical Care Survey** is a national CDC survey on emergency department visits across the United States, conducted annually from 1992 to 2022.  
Each year, approximately 500 hospitals provided data on a sample of ~16,000 ED visits.

## How to download the data
Pre-built Stata datasets (`.dta`) are the most convenient format for Python.  
Download them from the Stata folder on the CDC FTP server:  
**https://ftp.cdc.gov/pub/health_statistics/nchs/Dataset_Documentation/NHAMCS/stata/**

Files to download and decompress into `dataset/raw/nhamcs/`:


## Expected folder structure
```
dataset/raw/nhamcs/
├── ed2017-stata.dta   (or whatever name it has after decompression)
├── ed2018-stata.dta
├── ed2019-stata.dta
├── ed2020-stata.dta
├── ed2021-stata.dta
├── ed2022-stata.dta
└── ed2022              (optional: original ASCII file)
```

In [9]:
import pandas as pd
import numpy as np
import os
import json
import glob

In [10]:
# === Paths ===
# Modify BASE_DIR if the folder structure is different
BASE_DIR = "dataset"
NHAMCS_RAW = os.path.join(BASE_DIR, "raw", "nhamcs")
PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
SCHEMA_PATH = os.path.join(BASE_DIR, "schema_target.json")

os.makedirs(NHAMCS_RAW, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Load the target schema (generated in Section 3 of data_integration.ipynb)
with open(SCHEMA_PATH, "r") as f:
    schema_target = json.load(f)

---
## Step 1: Loading Stata Files (.dta)

We look for all `.dta` files in the NHAMCS folder and load them.  
If no `.dta` files are found but the ASCII file `ed2022` exists, we read it as a fallback.

In [11]:
def load_nhamcs_stata(nhamcs_dir):
    """
    Searches for all .dta files in the folder and loads them into a list of DataFrames.
    Adds a 'nhamcs_year' column to track the year of origin.
    """
    dta_files = sorted(glob.glob(os.path.join(nhamcs_dir, "*.dta")))
    
    if not dta_files:
        print(f"WARNING: no .dta files found in {nhamcs_dir}")
        print("Download the Stata files from:")
        print("https://ftp.cdc.gov/pub/health_statistics/nchs/Dataset_Documentation/NHAMCS/stata/")
        return None
    
    frames = []
    for filepath in dta_files:
        filename = os.path.basename(filepath)
        print(f"Loading: {filename}...", end=" ")
        
        df = pd.read_stata(filepath, convert_categoricals=False)
        # Normalize column names to UPPERCASE (some years use lowercase)
        df.columns = [c.upper() for c in df.columns]
        
        # Extract year from filename or YEAR column
        if 'YEAR' in df.columns:
            year = int(df['YEAR'].iloc[0])
        else:
            # Try to extract from filename (e.g. ed2022-stata.dta -> 2022)
            import re
            match = re.search(r'(20\d{2})', filename)
            year = int(match.group(1)) if match else 0
        
        df['nhamcs_year'] = year
        print(f"{len(df)} records, year {year}")
        frames.append(df)
    
    combined = pd.concat(frames, ignore_index=True)
    print(f"\nTotal records loaded: {len(combined)}")
    return combined


def load_nhamcs_ascii_2022(filepath):
    """
    Fallback: reads the 2022 fixed-width ASCII file.
    Column positions come from the codebook doc22-ed-508.pdf (Section II).
    WARNING: these positions are valid ONLY for 2022.
    """
    colspecs = [
        (0, 2),    # VMONTH
        (2, 3),    # VDAYR
        (3, 7),    # ARRTIME
        (7, 11),   # WAITTIME
        (11, 15),  # LOV
        (15, 18),  # AGE
        (18, 19),  # AGER
        (24, 25),  # SEX
        (32, 34),  # ARREMS
        (45, 47),  # PAYTYPER
        (47, 51),  # TEMPF (implied decimal between 3rd and 4th digit)
        (51, 54),  # PULSE
        (54, 57),  # RESPR
        (57, 60),  # BPSYS
        (60, 63),  # BPDIAS
        (63, 66),  # POPCT
        (66, 68),  # IMMEDR
        (68, 70),  # PAINSCALE
    ]
    names = [
        'VMONTH', 'VDAYR', 'ARRTIME', 'WAITTIME', 'LOV',
        'AGE', 'AGER', 'SEX', 'ARREMS', 'PAYTYPER',
        'TEMPF', 'PULSE', 'RESPR', 'BPSYS', 'BPDIAS', 'POPCT',
        'IMMEDR', 'PAINSCALE',
    ]
    
    df = pd.read_fwf(filepath, colspecs=colspecs, names=names)
    df['nhamcs_year'] = 2022
    print(f"Loaded ASCII file 2022: {len(df)} records")
    return df

print("Loading functions defined.")

Loading functions defined.


In [12]:
# === Load data ===
# Try Stata files first, then ASCII fallback
nhamcs_raw = load_nhamcs_stata(NHAMCS_RAW)

if nhamcs_raw is None:
    # Fallback: try the 2022 ASCII file
    ascii_path = os.path.join(NHAMCS_RAW, "ed2022")
    if os.path.exists(ascii_path):
        print(f"\nStata files not found. Using 2022 ASCII file as fallback.")
        nhamcs_raw = load_nhamcs_ascii_2022(ascii_path)
    else:
        raise FileNotFoundError(
            f"No NHAMCS data found in {NHAMCS_RAW}. "
            "Download .dta files from https://ftp.cdc.gov/pub/health_statistics/nchs/Dataset_Documentation/NHAMCS/stata/"
        )

print(f"\nTotal shape: {nhamcs_raw.shape}")
print(f"Years present: {sorted(nhamcs_raw['nhamcs_year'].unique())}")
print(f"Records per year:")
print(nhamcs_raw['nhamcs_year'].value_counts().sort_index())

Loading: ED2018-stata.dta... 

C:\Users\simop\AppData\Local\Temp\ipykernel_17832\3370590741.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['nhamcs_year'] = year


20291 records, year 2018
Loading: ED2019-stata.dta... 

C:\Users\simop\AppData\Local\Temp\ipykernel_17832\3370590741.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['nhamcs_year'] = year


19481 records, year 2019
Loading: ed2017-stata.dta... 

...

14860 records, year 2020
Loading: ed2021-stata.dta... 

C:\Users\simop\AppData\Local\Temp\ipykernel_17832\3370590741.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['nhamcs_year'] = year


16207 records, year 2021
Loading: ed2022-stata.dta... 

C:\Users\simop\AppData\Local\Temp\ipykernel_17832\3370590741.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['nhamcs_year'] = year


16025 records, year 2022

Total records loaded: 103573

Total shape: (103573, 958)
Years present: [2017, 2018, 2019, 2020, 2021, 2022]
Records per year:
nhamcs_year
2017    16709
2018    20291
2019    19481
2020    14860
2021    16207
2022    16025
Name: count, dtype: int64


---
## Step 2: Quick Exploration of Raw Data

We check key variables before transforming.

In [13]:
# Normalize column names to UPPERCASE (for safety, some .dta files use lowercase)
nhamcs_raw.columns = [c.upper() for c in nhamcs_raw.columns]

# Variables of interest
key_vars = ['AGE', 'SEX', 'IMMEDR', 'TEMPF', 'PULSE', 'RESPR', 
            'BPSYS', 'BPDIAS', 'POPCT', 'PAINSCALE', 'LOV',
            'ARREMS', 'PAYTYPER', 'VDAYR', 'VMONTH', 'ARRTIME',
            'WAITTIME', 'NHAMCS_YEAR']

# Show only columns that actually exist
available_keys = [v for v in key_vars if v in nhamcs_raw.columns]
missing_keys = [v for v in key_vars if v not in nhamcs_raw.columns]
if missing_keys:
    print(f"WARNING: columns not found: {missing_keys}")

print("=== IMMEDR Distribution (triage acuity) ===")
if 'IMMEDR' in nhamcs_raw.columns:
    print(nhamcs_raw['IMMEDR'].value_counts().sort_index())
    usable_mask = nhamcs_raw['IMMEDR'].between(1, 5)
    print(f"\nRecords with valid triage (1-5): {usable_mask.sum()} / {len(nhamcs_raw)} ({100*usable_mask.mean():.1f}%)")

print("\n=== SEX Distribution ===")
if 'SEX' in nhamcs_raw.columns:
    print(nhamcs_raw['SEX'].value_counts().sort_index())

print("\n=== AGE (summary statistics) ===")
if 'AGE' in nhamcs_raw.columns:
    print(nhamcs_raw['AGE'].describe())

=== IMMEDR Distribution (triage acuity) ===
IMMEDR
-9     3717
-8    22122
 0     2797
 1      991
 2    10136
 3    35327
 4    20815
 5     3065
 7     4603
Name: count, dtype: int64

Records with valid triage (1-5): 70334 / 103573 (67.9%)

=== SEX Distribution ===
SEX
1    55560
2    48013
Name: count, dtype: int64

=== AGE (summary statistics) ===
count    103573.000000
mean         39.258484
std          24.533938
min           0.000000
25%          20.000000
50%          37.000000
75%          58.000000
max          94.000000
Name: AGE, dtype: float64


## Step 3: Cleaning and Transformation

This is the core part. We take the raw NHAMCS data and transform it to conform to the Triagegeist schema.

### Main operations:

1. **Filter**: keep only records with IMMEDR between 1 and 5 (valid triage)
2. **Unit conversion**: temperature from Fahrenheit to Celsius
3. **Category mapping**: sex, day of week, payment type, arrival mode
4. **Comorbidity extraction**: 15 chronic condition flags mapped to hx_* columns
5. **Disposition reconstruction**: 10 binary outcome variables combined into a single categorical column
6. **Feature derivation**: arrival_hour from ARRTIME, ed_los_hours from LOV, MAP, pulse pressure, shock index
7. **Column renaming**: NHAMCS names -> target schema names

### NHAMCS -> Triagegeist Schema Variable Mapping

#### Vital Signs & Triage

| NHAMCS | Triagegeist | Transformation |
|--------|-------------|----------------|
| `IMMEDR` | `triage_acuity` | Direct (1-5). Filter out-of-range values. |
| `TEMPF` | `temperature_c` | Implied decimal in ASCII files, then (F-32)*5/9 |
| `PULSE` | `heart_rate` | Direct (exclude 998=Doppler) |
| `RESPR` | `respiratory_rate` | Direct |
| `BPSYS` | `systolic_bp` | Direct |
| `BPDIAS` | `diastolic_bp` | Direct (exclude 998=Palp/Doppler) |
| `POPCT` | `spo2` | Direct |
| `PAINSCALE` | `pain_score` | Direct (0-10) |

#### Demographics & Contextual

| NHAMCS | Triagegeist | Transformation |
|--------|-------------|----------------|
| `AGE` | `age` | Direct |
| `SEX` | `sex` | 1->F, 2->M |
| `ARREMS` + `AMBTRANSFER` | `arrival_mode` | 1+1->transfer, 1+2->ambulance, 2->walk-in |
| `PAYTYPER` | `insurance_type` | 1->private, 2/3->public, 5->self-pay, rest->other |
| `VDAYR` | `arrival_day` | 1->Sunday, ... 7->Saturday |
| `VMONTH` | `arrival_month` | Direct |
| `ARRTIME` | `arrival_hour` | Extract hour from military time (HHMM) |
| `LOV` | `ed_los_hours` | Minutes -> hours. Not available in 2017. |

#### Comorbidities

| NHAMCS | Triagegeist | NHAMCS | Triagegeist |
|--------|-------------|--------|-------------|
| `HTN` | `hx_hypertension` | `CANCER` | `hx_malignancy` |
| `DIABTYP1` | `hx_diabetes_type1` | `OBESITY` | `hx_obesity` |
| `DIABTYP2` | `hx_diabetes_type2` | `DEPRN` | `hx_depression` |
| `ASTHMA` | `hx_asthma` | `ALZHD` | `hx_dementia` |
| `COPD` | `hx_copd` | `EDHIV` | `hx_hiv` |
| `CHF` | `hx_heart_failure` | `SUBSTAB` | `hx_substance_use_disorder` |
| `CKD` | `hx_ckd` | `CEBVD` | `hx_stroke_prior` |
| `CAD` | `hx_coronary_artery_disease` | | |

#### Counts & Disposition

| NHAMCS | Triagegeist | Transformation |
|--------|-------------|----------------|
| `TOTCHRON` | `num_comorbidities` | Direct |
| `NUMMED` | `num_active_medications` | Direct |
| `ADMITHOS`, `DIEDED`, `LWBS`, ... | `disposition` | 10 binary vars combined with priority logic |

#### Derived Features (computed, not from raw NHAMCS columns)

| Triagegeist | Formula |
|-------------|---------|
| `mean_arterial_pressure` | (SBP + 2*DBP) / 3 |
| `pulse_pressure` | SBP - DBP |
| `shock_index` | HR / SBP |
| `age_group` | Derived from age: <18 pediatric, 18-39 young_adult, 40-64 middle_aged, 65+ senior |


In [14]:
def clean_nhamcs(df_raw):
    """
    Takes the raw NHAMCS DataFrame (single year or multi-year)
    and transforms it to the Triagegeist schema.

    Covers: vitals, demographics, triage, comorbidities, disposition,
    medication count, derived features (MAP, pulse pressure, shock index,
    NEWS2 approximation), pain_location from RFV13D, and categorical mappings.

    Tested against all .do files from 2018-2022: all mapped variables
    are consistently available across all 6 years.

    Returns:
        df_clean: DataFrame conforming to the target schema
        stats: dict with cleaning process statistics
    """
    df = df_raw.copy()
    # Normalize columns to uppercase (some .dta years use lowercase)
    df.columns = [c.upper() for c in df.columns]
    stats = {"records_in": len(df)}

    # =================================================================
    # 0. DROP 2017 RECORDS (LOV is a placeholder in that year)
    # =================================================================
    year_col = 'NHAMCS_YEAR' if 'NHAMCS_YEAR' in df.columns else 'YEAR'
    if year_col in df.columns:
        year_vals = pd.to_numeric(df[year_col], errors='coerce')
        before_drop = len(df)
        df = df[year_vals != 2017].copy()
        stats["records_dropped_2017"] = before_drop - len(df)
        if stats["records_dropped_2017"] > 0:
            print(f"Dropped {stats['records_dropped_2017']} records from 2017 (LOV not available)")
    else:
        stats["records_dropped_2017"] = 0

    # =================================================================
    # 1. FILTER: keep only records with valid triage (1-5)
    # =================================================================
    # IMMEDR encoding:
    #   -9 = Blank, -8 = Unknown, 0 = No triage but ESA conducts triage,
    #   1-5 = Triage levels (1=Immediate...5=Nonurgent),
    #   7 = ESA does not conduct triage
    if 'IMMEDR' not in df.columns:
        print("WARNING: IMMEDR column not found!")
        return None, stats

    df = df[df['IMMEDR'].between(1, 5)].copy()
    stats["records_after_triage_filter"] = len(df)
    stats["triage_filtered_out"] = stats["records_in"] - stats.get("records_dropped_2017", 0) - len(df)

    # =================================================================
    # 2. NUMERICAL CONVERSIONS — VITAL SIGNS
    # =================================================================

    # --- Temperature: Fahrenheit -> Celsius ---
    # In ASCII files, TEMPF has an implied decimal (0984 = 98.4F).
    # In Stata .dta files the value may already be correct.
    # We check: if median > 500, there is an implied decimal.
    if 'TEMPF' in df.columns:
        tempf = pd.to_numeric(df['TEMPF'], errors='coerce')
        if tempf.median() > 500:
            tempf = tempf / 10.0
        tempf = tempf.where(tempf.between(80, 110))
        df['temperature_c'] = ((tempf - 32) * 5 / 9).round(1)

    # --- Heart rate ---
    if 'PULSE' in df.columns:
        pulse = pd.to_numeric(df['PULSE'], errors='coerce')
        # 998 = Doppler (not a valid heart rate value)
        df['heart_rate'] = pulse.where(pulse.between(20, 250))

    # --- Respiratory rate ---
    if 'RESPR' in df.columns:
        respr = pd.to_numeric(df['RESPR'], errors='coerce')
        df['respiratory_rate'] = respr.where(respr.between(4, 60))

    # --- Systolic blood pressure ---
    if 'BPSYS' in df.columns:
        bpsys = pd.to_numeric(df['BPSYS'], errors='coerce')
        df['systolic_bp'] = bpsys.where(bpsys.between(30, 300))

    # --- Diastolic blood pressure ---
    if 'BPDIAS' in df.columns:
        bpdias = pd.to_numeric(df['BPDIAS'], errors='coerce')
        # 998 = Palp/Doppler
        df['diastolic_bp'] = bpdias.where(bpdias.between(10, 200))

    # --- Oxygen saturation ---
    if 'POPCT' in df.columns:
        popct = pd.to_numeric(df['POPCT'], errors='coerce')
        df['spo2'] = popct.where(popct.between(50, 100))

    # --- Pain scale ---
    if 'PAINSCALE' in df.columns:
        pain = pd.to_numeric(df['PAINSCALE'], errors='coerce')
        df['pain_score'] = pain.where(pain.between(0, 10))

    # --- ED length of stay: minutes -> hours ---
    if 'LOV' in df.columns:
        lov = pd.to_numeric(df['LOV'], errors='coerce')
        df['ed_los_hours'] = (lov.where(lov >= 0) / 60).round(2)

    # =================================================================
    # 3. TRIAGE ACUITY + DEMOGRAPHICS
    # =================================================================

    # --- Triage acuity: direct mapping ---
    df['triage_acuity'] = pd.to_numeric(df['IMMEDR'], errors='coerce').astype('Int64')

    # --- Age ---
    if 'AGE' in df.columns:
        df['age'] = pd.to_numeric(df['AGE'], errors='coerce')
        df['age'] = df['age'].where(df['age'].between(0, 120))

    # --- Sex ---
    # NHAMCS: 1=Female, 2=Male
    if 'SEX' in df.columns:
        df['sex'] = pd.to_numeric(df['SEX'], errors='coerce').map({1: 'F', 2: 'M'})

    # --- Age group (derived from age) ---
    if 'age' in df.columns:
        conditions = [
            df['age'] < 18,
            df['age'].between(18, 39),
            df['age'].between(40, 64),
            df['age'] >= 65,
        ]
        choices = ['pediatric', 'young_adult', 'middle_aged', 'senior']
        df['age_group'] = np.select(conditions, choices, default='unknown')
        df.loc[df['age_group'] == 'unknown', 'age_group'] = np.nan

    # =================================================================
    # 4. CATEGORICAL MAPPINGS
    # =================================================================

    # --- Arrival mode ---
    # ARREMS: 1=Ambulance Yes, 2=Ambulance No
    # AMBTRANSFER: 1=Transferred from another hospital, 2=No
    if 'ARREMS' in df.columns:
        arrems = pd.to_numeric(df['ARREMS'], errors='coerce')
        df['arrival_mode'] = arrems.map({1: 'ambulance', 2: 'walk-in'})
        # Refine: if ambulance AND transferred from another facility -> 'transfer'
        if 'AMBTRANSFER' in df.columns:
            ambtrans = pd.to_numeric(df['AMBTRANSFER'], errors='coerce')
            transfer_mask = (arrems == 1) & (ambtrans == 1)
            df.loc[transfer_mask, 'arrival_mode'] = 'transfer'

    # --- Day of week ---
    # VDAYR: 1=Sunday, 2=Monday, ..., 7=Saturday
    if 'VDAYR' in df.columns:
        day_map = {
            1: 'Sunday', 2: 'Monday', 3: 'Tuesday', 4: 'Wednesday',
            5: 'Thursday', 6: 'Friday', 7: 'Saturday'
        }
        df['arrival_day'] = pd.to_numeric(df['VDAYR'], errors='coerce').map(day_map)

    # --- Month ---
    if 'VMONTH' in df.columns:
        df['arrival_month'] = pd.to_numeric(df['VMONTH'], errors='coerce')
        df['arrival_month'] = df['arrival_month'].where(df['arrival_month'].between(1, 12))

    # --- Arrival hour: extract from military time HHMM ---
    if 'ARRTIME' in df.columns:
        arrtime = pd.to_numeric(df['ARRTIME'], errors='coerce')
        df['arrival_hour'] = (arrtime.where(arrtime.between(0, 2359)) // 100).astype('Int64')

    # --- Payment / insurance type ---
    # PAYTYPER: 1=Private, 2=Medicare, 3=Medicaid/CHIP, 4=Workers comp,
    #           5=Self-pay, 6=No charge/Charity, 7=Other
    if 'PAYTYPER' in df.columns:
        pay_map = {
            1: 'private',
            2: 'public',     # Medicare -> public
            3: 'public',     # Medicaid/CHIP -> public
            4: 'other',      # Workers comp -> other
            5: 'self-pay',
            6: 'other',      # No charge/Charity -> other
            7: 'other',
        }
        df['insurance_type'] = pd.to_numeric(df['PAYTYPER'], errors='coerce').map(pay_map)

    # =================================================================
    # 5. COMORBIDITIES (binary 0/1, consistent across 2018-2022)
    # =================================================================
    # Verified present in all .do files: ed2017.do through ed2022.do
    # NHAMCS uses 0=No, 1=Yes for all chronic condition checkboxes.
    # We convert: valid 0/1 -> keep as is; -9/-8/other -> NaN

    comorbidity_map = {
        'HTN':      'hx_hypertension',
        'DIABTYP1': 'hx_diabetes_type1',
        'DIABTYP2': 'hx_diabetes_type2',
        'ASTHMA':   'hx_asthma',
        'COPD':     'hx_copd',
        'CHF':      'hx_heart_failure',
        'CKD':      'hx_ckd',
        'CAD':      'hx_coronary_artery_disease',
        'CEBVD':    'hx_stroke_prior',
        'CANCER':   'hx_malignancy',
        'OBESITY':  'hx_obesity',
        'DEPRN':    'hx_depression',
        'ALZHD':    'hx_dementia',
        'EDHIV':    'hx_hiv',
        'SUBSTAB':  'hx_substance_use_disorder',
    }

    for nhamcs_col, tg_col in comorbidity_map.items():
        if nhamcs_col in df.columns:
            val = pd.to_numeric(df[nhamcs_col], errors='coerce')
            # Keep only valid 0/1 values; everything else becomes NaN
            df[tg_col] = val.where(val.isin([0, 1]))

    # --- Total number of chronic conditions ---
    if 'TOTCHRON' in df.columns:
        totchron = pd.to_numeric(df['TOTCHRON'], errors='coerce')
        df['num_comorbidities'] = totchron.where(totchron >= 0)

    # --- Number of medications ---
    if 'NUMMED' in df.columns:
        nummed = pd.to_numeric(df['NUMMED'], errors='coerce')
        df['num_active_medications'] = nummed.where(nummed >= 0)

    # =================================================================
    # 6. DISPOSITION (outcome_only — for validation, NOT training)
    # =================================================================
    # Combine multiple binary disposition columns into a single categorical.
    # Priority order: deceased > admitted > transferred > observation
    # > lama > lwbs > discharged
    # All disposition vars are binary (0=No, 1=Yes), consistent 2017-2022.

    disp_cols = {
        'DIEDED': 'deceased', 'DOA': 'deceased',
        'ADMITHOS': 'admitted', 'OBSHOS': 'admitted',
        'TRANOTH': 'transferred', 'TRANPSYC': 'transferred', 'TRANNH': 'transferred',
        'OBSDIS': 'observation',
        'LEFTAMA': 'lama',
        'LWBS': 'lwbs',
    }

    available_disp = [c for c in disp_cols.keys() if c in df.columns]
    if available_disp:
        df['disposition'] = 'discharged'  # default
        # Apply in reverse priority order (last write wins for higher priority)
        priority_order = ['lwbs', 'lama', 'observation', 'transferred', 'admitted', 'deceased']
        for priority_label in priority_order:
            matching_cols = [c for c, label in disp_cols.items() if label == priority_label and c in df.columns]
            for col in matching_cols:
                val = pd.to_numeric(df[col], errors='coerce')
                df.loc[val == 1, 'disposition'] = priority_label

    # =================================================================
    # 7. PAIN LOCATION (from RFV13D — broader reason-for-visit category)
    # =================================================================
    # RFV13D is the 3-digit grouped version of the Reason for Visit code.
    # The first 2 digits (prefix = code // 100) map to body systems in the
    # CDC's Reason for Visit Classification (RVC). The mapping below follows
    # the RVC module structure verified in Appendix II of doc22-ed-508.pdf.
    #
    # Triagegeist pain_location values:
    #   chest, abdomen, head, limb, pelvis, back, neck, diffuse, none

    def rfv13d_to_pain_location(code):
        if pd.isna(code) or code <= 0:
            return np.nan
        c = int(code)
        prefix = c // 100  # first 2 digits: 10, 11, 12, ..., 59, ...

        # --- Module 1: Symptoms (1xxx) ---
        if prefix == 10:  return 'diffuse'    # general symptoms (fever, fatigue, pain NOS)
        if prefix == 11:  return 'none'       # skin, hair, nails (rash, wound appearance)
        if prefix == 12:  return 'limb'       # musculoskeletal (joint/limb pain, swelling)
        if prefix == 13:  return 'chest'      # respiratory (cough, dyspnea, chest tightness)
        if prefix == 14:  return 'chest'      # cardiovascular (chest pain, palpitations)
        if prefix == 15:  return 'abdomen'    # digestive (abdominal pain, nausea, vomiting)
        if prefix == 16:  return 'pelvis'     # urogenital (urinary, reproductive symptoms)
        if prefix == 17:  return 'head'       # nervous system (headache, dizziness, numbness)
        if prefix == 18:  return 'head'       # eyes, ears, nose, mouth, throat
        if prefix == 19:  return 'none'       # psych/behavioral, general unspecified

        # --- Module 2: Diseases (2xxx) ---
        if prefix == 20:  return 'none'       # infectious diseases
        if prefix == 21:  return 'none'       # neoplasms
        if prefix == 22:  return 'none'       # endocrine/metabolic
        if prefix == 23:  return 'none'       # blood diseases
        if prefix == 24:  return 'none'       # mental disorders
        if prefix == 25:  return 'head'       # nervous system diseases
        if prefix == 26:  return 'none'       # circulatory diseases
        if prefix == 27:  return 'chest'      # respiratory diseases
        if prefix == 28:  return 'abdomen'    # digestive diseases
        if prefix == 29:  return 'none'       # other diseases

        # --- Module 3-4: Diagnostic/screening + Treatment (3xxx-4xxx) ---
        if 30 <= prefix <= 48:  return 'none'

        # --- Module 5: Injuries (5xxx) — mapped by injury body site ---
        if prefix == 50:  return 'head'       # head and face injuries
        if prefix == 51:  return 'neck'       # neck injuries
        if prefix == 52:  return 'limb'       # upper extremity (arm, hand, shoulder)
        if prefix == 53:  return 'chest'      # trunk/chest injuries
        if prefix == 54:  return 'back'       # back and spine injuries
        if prefix == 55:  return 'limb'       # lower extremity (leg, foot, hip, knee)
        if prefix == 56:  return 'diffuse'    # multiple body sites
        if prefix == 57:  return 'none'       # poisoning, adverse effects
        if prefix == 58:  return 'none'       # complications, other injury effects
        if prefix == 59:  return 'none'       # unspecified injury site

        # --- Module 6-8: Test results, admin, other ---
        return 'none'

    if 'RFV13D' in df.columns:
        rfv13d = pd.to_numeric(df['RFV13D'], errors='coerce')
        df['pain_location'] = rfv13d.apply(rfv13d_to_pain_location)
        stats["pain_location_mapped"] = df['pain_location'].notna().sum()
        stats["pain_location_distribution"] = df['pain_location'].value_counts().to_dict()

    # =================================================================
    # 8. NEWS2 APPROXIMATE SCORE
    # =================================================================
    # NEWS2 is based on 6 physiological parameters + supplemental O2.
    # We have 5 of 6: RR, SpO2, SBP, HR, Temperature.
    # Missing: level of consciousness (ACVPU scale) — not collected in NHAMCS.
    # Standard practice: assume score 0 for missing consciousness (= Alert).
    # Also missing: supplemental O2 flag — assume 0 (no supplemental O2).
    #
    # This gives a NEWS2 that systematically underestimates by 0-5 points
    # for patients who are not alert or on supplemental oxygen.
    # We flag this with a note in the schema.
    #
    # Scoring table (Royal College of Physicians, 2017):
    # https://www.rcp.ac.uk/projects/outputs/national-early-warning-score-news-2
    #
    # Each parameter scores 0-3, total range 0-20 (we cap at 0-15 without
    # consciousness and supplemental O2).

    def compute_news2_approx(row):
        score = 0
        has_any = False

        # --- Respiratory rate ---
        rr = row.get('respiratory_rate', np.nan)
        if pd.notna(rr):
            has_any = True
            if rr <= 8:                score += 3
            elif rr <= 11:             score += 1
            elif rr <= 20:             score += 0
            elif rr <= 24:             score += 2
            else:                      score += 3  # >=25

        # --- SpO2 (Scale 1 — standard, not COPD) ---
        spo2 = row.get('spo2', np.nan)
        if pd.notna(spo2):
            has_any = True
            if spo2 <= 91:             score += 3
            elif spo2 <= 93:           score += 2
            elif spo2 <= 95:           score += 1
            else:                      score += 0  # >=96

        # --- Systolic blood pressure ---
        sbp = row.get('systolic_bp', np.nan)
        if pd.notna(sbp):
            has_any = True
            if sbp <= 90:              score += 3
            elif sbp <= 100:           score += 2
            elif sbp <= 110:           score += 1
            elif sbp <= 219:           score += 0
            else:                      score += 3  # >=220

        # --- Heart rate ---
        hr = row.get('heart_rate', np.nan)
        if pd.notna(hr):
            has_any = True
            if hr <= 40:               score += 3
            elif hr <= 50:             score += 1
            elif hr <= 90:             score += 0
            elif hr <= 110:            score += 1
            elif hr <= 130:            score += 2
            else:                      score += 3  # >=131

        # --- Temperature (Celsius) ---
        temp = row.get('temperature_c', np.nan)
        if pd.notna(temp):
            has_any = True
            if temp <= 35.0:           score += 3
            elif temp <= 36.0:         score += 1
            elif temp <= 38.0:         score += 0
            elif temp <= 39.0:         score += 1
            else:                      score += 2  # >=39.1

        # Consciousness: not available in NHAMCS, assumed Alert = 0
        # Supplemental O2: not available in NHAMCS, assumed No = 0

        if not has_any:
            return np.nan
        return score

    # Apply row-wise — this is slow but correct for ~60k rows
    required_vitals = ['respiratory_rate', 'spo2', 'systolic_bp', 'heart_rate', 'temperature_c']
    if all(col in df.columns for col in required_vitals):
        print("Computing approximate NEWS2 scores...")
        df['news2_score'] = df.apply(compute_news2_approx, axis=1)
        stats["news2_computed"] = df['news2_score'].notna().sum()
        stats["news2_mean"] = round(df['news2_score'].mean(), 2)
        print(f"  NEWS2 computed for {stats['news2_computed']} records (mean={stats['news2_mean']})")
    else:
        missing_vitals = [c for c in required_vitals if c not in df.columns]
        print(f"WARNING: Cannot compute NEWS2 — missing columns: {missing_vitals}")

    # =================================================================
    # 9. DERIVED FEATURES
    # =================================================================

    # Mean arterial pressure: (SBP + 2*DBP) / 3
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        df['mean_arterial_pressure'] = ((df['systolic_bp'] + 2 * df['diastolic_bp']) / 3).round(1)

    # Pulse pressure: SBP - DBP
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']

    # Shock index: HR / SBP
    if 'heart_rate' in df.columns and 'systolic_bp' in df.columns:
        df['shock_index'] = (df['heart_rate'] / df['systolic_bp']).round(3)
        df['shock_index'] = df['shock_index'].where(df['shock_index'].between(0.1, 5.0))

    # =================================================================
    # 10. GENERATE patient_id AND data_source
    # =================================================================

    # Create a unique ID with NHAMCS prefix + year + index
    year_col_final = 'NHAMCS_YEAR' if 'NHAMCS_YEAR' in df.columns else 'YEAR'
    if year_col_final in df.columns:
        years = pd.to_numeric(df[year_col_final], errors='coerce').fillna(0).astype(int)
    else:
        years = pd.Series([0] * len(df), index=df.index)

    df['patient_id'] = [
        f"NHAMCS_{y}_{i:06d}"
        for i, y in enumerate(years)
    ]
    df['data_source'] = 'nhamcs'

    # =================================================================
    # 11. FINAL COLUMN SELECTION
    # =================================================================

    # Keep only columns that exist in the target schema
    schema_cols = list(schema_target['columns'].keys())

    # Add schema columns we don't have as NaN
    for col in schema_cols:
        if col not in df.columns:
            df[col] = np.nan

    df_clean = df[schema_cols].copy()

    # --- Build stats report ---
    stats["records_out"] = len(df_clean)
    stats["columns_with_data"] = [c for c in schema_cols if df_clean[c].notna().any()]
    stats["columns_all_nan"] = [c for c in schema_cols if df_clean[c].isna().all()]

    return df_clean, stats


In [15]:
# === Run the cleaning ===
nhamcs_clean, nhamcs_stats = clean_nhamcs(nhamcs_raw)

print("=== NHAMCS Cleaning Statistics ===")
print(f"Records in:                    {nhamcs_stats['records_in']}")
if 'records_dropped_2017' in nhamcs_stats:
    print(f"Dropped (2017, no LOV):        {nhamcs_stats['records_dropped_2017']}")
print(f"Filtered out (invalid triage): {nhamcs_stats['triage_filtered_out']}")
print(f"Records out:                   {nhamcs_stats['records_out']}")
print(f"\nColumns with real NHAMCS data ({len(nhamcs_stats['columns_with_data'])}):\n  {nhamcs_stats['columns_with_data']}")
print(f"\nColumns left as NaN ({len(nhamcs_stats['columns_all_nan'])}):\n  {nhamcs_stats['columns_all_nan']}")
if 'news2_mean' in nhamcs_stats:
    print(f"\nNEWS2 approx mean:             {nhamcs_stats['news2_mean']}")
if 'pain_location_distribution' in nhamcs_stats:
    print(f"\nPain location distribution:")
    for loc, count in sorted(nhamcs_stats['pain_location_distribution'].items()):
        print(f"  {loc:12s}: {count}")

Dropped 16709 records from 2017 (LOV not available)
Computing approximate NEWS2 scores...
  NEWS2 computed for 57610 records (mean=1.55)
=== NHAMCS Cleaning Statistics ===
Records in:                    103573
Dropped (2017, no LOV):        16709
Filtered out (invalid triage): 28740
Records out:                   58124

Columns with real NHAMCS data (42):
  ['patient_id', 'triage_acuity', 'disposition', 'ed_los_hours', 'arrival_mode', 'arrival_day', 'age_group', 'sex', 'insurance_type', 'pain_location', 'systolic_bp', 'diastolic_bp', 'mean_arterial_pressure', 'pulse_pressure', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2', 'pain_score', 'shock_index', 'news2_score', 'age', 'arrival_hour', 'arrival_month', 'num_active_medications', 'num_comorbidities', 'hx_hypertension', 'hx_diabetes_type2', 'hx_diabetes_type1', 'hx_asthma', 'hx_copd', 'hx_heart_failure', 'hx_ckd', 'hx_malignancy', 'hx_obesity', 'hx_depression', 'hx_dementia', 'hx_hiv', 'hx_substance_use_disorder', 'hx_coron

In [16]:
# === Quality checks ===
print("=== triage_acuity Distribution ===")
print(nhamcs_clean['triage_acuity'].value_counts().sort_index())

print("\n=== sex Distribution ===")
print(nhamcs_clean['sex'].value_counts())

print("\n=== Vital Signs Statistics ===")
vitals = ['temperature_c', 'heart_rate', 'respiratory_rate', 
          'systolic_bp', 'diastolic_bp', 'spo2', 'pain_score']
print(nhamcs_clean[vitals].describe().round(1).to_string())

print("\n=== insurance_type Distribution ===")
print(nhamcs_clean['insurance_type'].value_counts())

print("\n=== NaN per column (top 10) ===")
nan_pct = (nhamcs_clean.isna().mean() * 100).sort_values(ascending=False)
print(nan_pct.head(10).round(1).to_string())

=== triage_acuity Distribution ===
triage_acuity
1      846
2     8597
3    29568
4    16715
5     2398
Name: count, dtype: Int64

=== sex Distribution ===
sex
F    31086
M    27038
Name: count, dtype: int64

=== Vital Signs Statistics ===
       temperature_c  heart_rate  respiratory_rate  systolic_bp  diastolic_bp     spo2  pain_score
count        56079.0     56347.0           56325.0      52970.0       52898.0  55769.0     43319.0
mean            36.8        91.9              19.3        134.9          79.1     97.7         4.4
std              0.6        22.7               4.7         24.4          14.9      3.0         3.7
min             28.9        20.0               4.0         43.0          21.0     52.0         0.0
25%             36.6        77.0              16.0        118.0          69.0     97.0         0.0
50%             36.8        88.0              18.0        132.0          78.0     98.0         5.0
75%             37.0       103.0              20.0        149.0    

---
## Step 4: Save

In [17]:
# === Save cleaned dataset ===
output_path = os.path.join(PROCESSED_DIR, "nhamcs_clean.csv")
nhamcs_clean.to_csv(output_path, index=False)

print(f"Cleaned NHAMCS dataset saved to: {output_path}")
print(f"Shape: {nhamcs_clean.shape}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.1f} MB")

nhamcs_clean.head(3)

Cleaned NHAMCS dataset saved to: dataset\processed\nhamcs_clean.csv
Shape: (58124, 66)
File size: 11.4 MB


,patient_id,triage_acuity,disposition,ed_los_hours,arrival_mode,arrival_day,arrival_season,shift,age_group,sex,...,hx_hiv,hx_coagulopathy,hx_immunosuppressed,hx_pregnant,hx_substance_use_disorder,hx_coronary_artery_disease,hx_stroke_prior,hx_peripheral_vascular_disease,chief_complaint_raw,data_source
0,NHAMCS_2018_000000,4,discharged,1.55,walk-in,Sunday,NaN,NaN,pediatric,M,...,0,NaN,NaN,NaN,0,0,0,NaN,NaN,nhamcs
1,NHAMCS_2018_000001,4,discharged,0.80,walk-in,Sunday,NaN,NaN,pediatric,F,...,0,NaN,NaN,NaN,0,0,0,NaN,NaN,nhamcs
2,NHAMCS_2018_000002,4,discharged,1.65,walk-in,Friday,NaN,NaN,pediatric,M,...,0,NaN,NaN,NaN,0,0,0,NaN,NaN,nhamcs


---
## Important Notes on NHAMCS

### What is available (42 columns with real data)
- **Vital signs**: temperature, HR, RR, BP, SpO2
- **Triage level**: IMMEDR, 1-5 scale equivalent to ESI
- **Pain scale**: 0-10
- **Demographics**: age, sex, ethnicity
- **Payment type / insurance**
- **Arrival mode**: ambulance, walk-in, or transfer (refined using AMBTRANSFER)
- **ED length of stay** (2018-2022 only)
- **15 comorbidities**: hypertension, diabetes (type 1 & 2), asthma, COPD, heart failure, CKD, CAD, stroke, cancer, obesity, depression, dementia, HIV, substance use disorder
- **Comorbidity count** (TOTCHRON) and **medication count** (NUMMED)
- **Disposition**: reconstructed from 10 binary outcome variables (admitted, discharged, deceased, transferred, observation, lama, lwbs)
- **Pain location**: mapped from RFV13D (broader reason-for-visit category) using the CDC's Reason for Visit Classification body system prefixes. Maps to the 9 Triagegeist categories: chest, abdomen, head, limb, pelvis, back, neck, diffuse, none.
- **NEWS2 score (approximate)**: computed from 5 of 6 NEWS2 parameters (RR, SpO2, SBP, HR, temperature). Level of consciousness and supplemental oxygen are not available in NHAMCS and are assumed to score 0 (Alert, no O2). This produces a systematic underestimate of 0–5 points for patients who are not alert or on supplemental oxygen.
- **Other derived features**: MAP, pulse pressure, shock index, age group
- ~16,000 records/year, 5 years (2018-2022) = ~80,000 total records
- Of these, ~60-65% have valid triage = ~50,000 usable records

### What is NOT available (24 columns that will remain NaN)
- GCS (Glasgow Coma Scale) — not collected in NHAMCS, not derivable
- Weight, height, BMI
- Some specific comorbidities not tracked by NHAMCS: atrial fibrillation, liver disease, anxiety, epilepsy, hypothyroidism, hyperthyroidism, coagulopathy, immunosuppressed status, pregnancy status, peripheral vascular disease
- Prior ED visits / admissions count (SEEN72 is binary, not a count)
- Chief complaint as free text (NHAMCS uses coded reason-for-visit, not free text)
- Shift, season, transport_origin, mental_status_triage
- site_id, triage_nurse_id, language

### Considerations
- **2017 records are excluded** because LOV (length of visit) is a placeholder in that year, making ed_los_hours unavailable
- NHAMCS data is **American**, Triagegeist simulates a **Finnish** hospital network: possible distribution differences
- ~35-40% of records have missing or non-applicable triage: filtered out
- NHAMCS is a **weighted survey**: national estimates require weights (PATWT), but for ML training we ignore them
- Temperature is in **Fahrenheit**: converted to Celsius by clean_nhamcs()
- Comorbidity variables use NHAMCS encoding: 0=No, 1=Yes; values -9/-8 are converted to NaN
- Disposition is reconstructed with priority logic: deceased > admitted > transferred > observation > lama > lwbs > discharged
- NEWS2 is an **approximation**: missing consciousness and supplemental O2 parameters mean the score underestimates severity for confused/unconscious patients and those on oxygen. Consider adding a boolean flag column `news2_is_approximate` if downstream analysis needs to distinguish exact from approximate scores
- Pain location mapping from RFV13D is based on body system prefixes, not on the specific complaint text. A visit coded as "cough" (respiratory system) maps to "chest" even though the patient may not have chest pain. This is a reasonable proxy but not a direct equivalent of the Triagegeist `pain_location` field